In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Set random seed
np.random.seed(1234)

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

In [9]:
import os
# Load the preprocessed data

# Set the correct path to your data directory
data_dir = '../data'  # Adjust this path to where your CSV files are located

# Load train, validation, and test sets
X_train = pd.read_csv(os.path.join(data_dir, 'X_train_scaled.csv'))
Y_train = pd.read_csv(os.path.join(data_dir, 'y_train.csv')).squeeze()


X_val = pd.read_csv(os.path.join(data_dir, 'X_val_scaled.csv'))
Y_val = pd.read_csv(os.path.join(data_dir, 'y_val.csv')).squeeze()


X_test = pd.read_csv(os.path.join(data_dir, 'X_test_scaled.csv'))
Y_test = pd.read_csv(os.path.join(data_dir, 'y_test.csv')).squeeze()


print("Data loaded successfully!")
print(f"\nTraining set shape: {X_train.shape}")
print(f"Validation set shape: {X_val.shape}")
print(f"Test set shape: {X_test.shape}")
print(f"\nTraining labels distribution:\n{Y_train.value_counts()}")


Data loaded successfully!

Training set shape: (40251, 68)
Validation set shape: (13417, 68)
Test set shape: (13418, 68)

Training labels distribution:
readmitted
0    36581
1     3670
Name: count, dtype: int64


In [10]:
# NaNs check
print("\nChecking for NaNs in datasets:")
print(f"NaNs in X_train: {X_train.isna().sum().sum()}")
print(f"NaNs in Y_train: {Y_train.isna().sum()}")
print(f"NaNs in X_val: {X_val.isna().sum().sum()}")
print(f"NaNs in Y_val: {Y_val.isna().sum()}")
print(f"NaNs in X_test: {X_test.isna().sum().sum()}")
print(f"NaNs in Y_test: {Y_test.isna().sum()}")


Checking for NaNs in datasets:
NaNs in X_train: 0
NaNs in Y_train: 0
NaNs in X_val: 0
NaNs in Y_val: 0
NaNs in X_test: 0
NaNs in Y_test: 0


In [11]:
# Baseline Decision Tree Classifier
from sklearn.tree import DecisionTreeClassifier

# Decision Tree Classifier basic model
dt_clf = DecisionTreeClassifier(
    random_state=1234, 
    max_depth=28, 
    criterion='gini',
    min_samples_split=10)

# fit the model
dt_clf.fit(X_train, Y_train)
# Evaluate on validation set
val_accuracy = dt_clf.score(X_val, Y_val)
print(f"Validation Accuracy of Decision Tree: {val_accuracy:.4f}")

Validation Accuracy of Decision Tree: 0.8556


In [12]:
# Baseline XGBoost model with default hyperparameters (no scale_pos_weight)
xgb_baseline = XGBClassifier(
    random_state=1234,
    n_jobs=-1,
    eval_metric="logloss",
)
# fit the model
xgb_baseline.fit(X_train, Y_train)
# Evaluate on validation set
val_accuracy_xgb = xgb_baseline.score(X_val, Y_val)
print(f"Validation Accuracy of XGBoost Baseline: {val_accuracy_xgb:.4f}")

Validation Accuracy of XGBoost Baseline: 0.9077


In [13]:
# Baseline Logistic Regression model
from sklearn.linear_model import LogisticRegression
logreg_baseline = LogisticRegression(
    random_state=1234,
    class_weight='balanced',
    solver='liblinear',
    n_jobs=-1,
)

# fit the model
logreg_baseline.fit(X_train, Y_train)
# Evaluate on validation set
val_accuracy_logreg = logreg_baseline.score(X_val, Y_val)
print(f"Validation Accuracy of Logistic Regression: {val_accuracy_logreg:.4f}") 

/home/cpfrish/miniforge3/envs/mids207/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1216: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 16.
  warnings.warn(


Validation Accuracy of Logistic Regression: 0.6578


In [21]:
# Random Foreest Classifier basic model
rf_clf = RandomForestClassifier(
    n_estimators=10,
    max_depth=25,
    bootstrap=True,
    class_weight=None,
    max_leaf_nodes=None,
    criterion='gini',
    min_samples_leaf=1,
    min_samples_split=10,
    min_weight_fraction_leaf=0.0,
    random_state=1234,
    n_jobs=None,
    oob_score=False,
    verbose=0,
    warm_start=False,
)
# fit the model
rf_clf.fit(X_train, Y_train)
# Evaluate on validation set
val_accuracy_rf = rf_clf.score(X_val, Y_val)
print(f"Validation Accuracy of Random Forest: {val_accuracy_rf:.4f}")

Validation Accuracy of Random Forest: 0.9088


In [15]:
# Compute F1 Scores for all models
from sklearn.metrics import f1_score, classification_report
Y_val_pred = dt_clf.predict(X_val)
val_f1_dt = f1_score(Y_val, Y_val_pred)
print(f"\nValidation F1 Score of Decision Tree: {val_f1_dt:.4f}")
print("Classification Report for Decision Tree:\n", classification_report(Y_val, Y_val_pred))
Y_val_pred_xgb = xgb_baseline.predict(X_val)
val_f1_xgb = f1_score(Y_val, Y_val_pred_xgb)
print(f"\nValidation F1 Score of XGBoost Baseline: {val_f1_xgb:.4f}")
print("Classification Report for XGBoost Baseline:\n", classification_report(Y_val, Y_val_pred_xgb))
Y_val_pred_logreg = logreg_baseline.predict(X_val)
val_f1_logreg = f1_score(Y_val, Y_val_pred_logreg)
print(f"\nValidation F1 Score of Logistic Regression: {val_f1_logreg:.4f}")
print("Classification Report for Logistic Regression:\n", classification_report(Y_val, Y_val_pred_logreg))
Y_val_pred_rf = rf_clf.predict(X_val)
val_f1_rf = f1_score(Y_val, Y_val_pred_rf)      
print(f"\nValidation F1 Score of Random Forest: {val_f1_rf:.4f}")
print("Classification Report for Random Forest:\n", classification_report(Y_val, Y_val_pred_rf))    


Validation F1 Score of Decision Tree: 0.1077
Classification Report for Decision Tree:
               precision    recall  f1-score   support

           0       0.91      0.93      0.92     12194
           1       0.12      0.10      0.11      1223

    accuracy                           0.86     13417
   macro avg       0.52      0.51      0.51     13417
weighted avg       0.84      0.86      0.85     13417


Validation F1 Score of XGBoost Baseline: 0.0282
Classification Report for XGBoost Baseline:
               precision    recall  f1-score   support

           0       0.91      1.00      0.95     12194
           1       0.35      0.01      0.03      1223

    accuracy                           0.91     13417
   macro avg       0.63      0.51      0.49     13417
weighted avg       0.86      0.91      0.87     13417


Validation F1 Score of Logistic Regression: 0.2178
Classification Report for Logistic Regression:
               precision    recall  f1-score   support

         

In [16]:
# Applying SMOTE to handle class imbalance
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=1234)
X_train_smote, Y_train_smote = smote.fit_resample(X_train, Y_train)
print(f"\nAfter SMOTE, training set shape: {X_train_smote.shape}")
print(f"After SMOTE, training labels distribution:\n{Y_train_smote.value_counts()}")


After SMOTE, training set shape: (73162, 68)
After SMOTE, training labels distribution:
readmitted
0    36581
1    36581
Name: count, dtype: int64


In [22]:
# Retrain all models on SMOTE data and show f1 scores
# Decision Tree
dt_clf.fit(X_train_smote, Y_train_smote)
Y_val_pred_smote_dt = dt_clf.predict(X_val)
val_f1_smote_dt = f1_score(Y_val, Y_val_pred_smote_dt)
print(f"\nAfter SMOTE, Validation F1 Score of Decision Tree: {val_f1_smote_dt:.4f}")
print("Classification Report for Decision Tree after SMOTE:\n", classification_report(Y_val, Y_val_pred_smote_dt))
# XGBoost
xgb_baseline.fit(X_train_smote, Y_train_smote)
Y_val_pred_smote_xgb = xgb_baseline.predict(X_val)
val_f1_smote_xgb = f1_score(Y_val, Y_val_pred_smote_xgb)
print(f"\nAfter SMOTE, Validation F1 Score of XGBoost Baseline: {val_f1_smote_xgb:.4f}")
print("Classification Report for XGBoost Baseline after SMOTE:\n", classification_report(Y_val, Y_val_pred_smote_xgb))
# Logistic Regression
logreg_baseline.fit(X_train_smote, Y_train_smote)
Y_val_pred_smote_logreg = logreg_baseline.predict(X_val)
val_f1_smote_logreg = f1_score(Y_val, Y_val_pred_smote_logreg)
print(f"\nAfter SMOTE, Validation F1 Score of Logistic Regression: {val_f1_smote_logreg:.4f}")
print("Classification Report for Logistic Regression after SMOTE:\n", classification_report(Y_val, Y_val_pred_smote_logreg))
# Random Forest
rf_clf.fit(X_train_smote, Y_train_smote)
Y_val_pred_smote_rf = rf_clf.predict(X_val)
val_f1_smote_rf = f1_score(Y_val, Y_val_pred_smote_rf)      
print(f"\nAfter SMOTE, Validation F1 Score of Random Forest: {val_f1_smote_rf:.4f}")
print("Classification Report for Random Forest after SMOTE:\n", classification_report(Y_val, Y_val_pred_smote_rf))  


After SMOTE, Validation F1 Score of Decision Tree: 0.1102
Classification Report for Decision Tree after SMOTE:
               precision    recall  f1-score   support

           0       0.91      0.93      0.92     12194
           1       0.12      0.10      0.11      1223

    accuracy                           0.85     13417
   macro avg       0.52      0.51      0.51     13417
weighted avg       0.84      0.85      0.85     13417


After SMOTE, Validation F1 Score of XGBoost Baseline: 0.0254
Classification Report for XGBoost Baseline after SMOTE:
               precision    recall  f1-score   support

           0       0.91      1.00      0.95     12194
           1       0.44      0.01      0.03      1223

    accuracy                           0.91     13417
   macro avg       0.68      0.51      0.49     13417
weighted avg       0.87      0.91      0.87     13417



/home/cpfrish/miniforge3/envs/mids207/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1216: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 16.
  warnings.warn(



After SMOTE, Validation F1 Score of Logistic Regression: 0.1645
Classification Report for Logistic Regression after SMOTE:
               precision    recall  f1-score   support

           0       0.92      0.75      0.83     12194
           1       0.11      0.31      0.16      1223

    accuracy                           0.71     13417
   macro avg       0.51      0.53      0.50     13417
weighted avg       0.84      0.71      0.77     13417


After SMOTE, Validation F1 Score of Random Forest: 0.0807
Classification Report for Random Forest after SMOTE:
               precision    recall  f1-score   support

           0       0.91      0.98      0.94     12194
           1       0.20      0.05      0.08      1223

    accuracy                           0.89     13417
   macro avg       0.55      0.52      0.51     13417
weighted avg       0.85      0.89      0.87     13417

